# 00 — Configuração do Colab (Fato / Tese / Ruído)

Este notebook prepara o ambiente no Google Colab: monta o Drive, instala
as dependências e valida GPU, caminhos e dataset. **Rode-o primeiro.**

## 1. Montar o Google Drive e localizar o projeto

⚠️ **Suba o PROJETO INTEIRO para o Drive**, não só a pasta `Data`. A pasta
no Drive (por padrão `/content/drive/MyDrive/Fato_Tese_Ruido`) precisa conter:

```
Fato_Tese_Ruido/
├── requirements.txt
├── src/ftr/...        ← o módulo 'ftr' (sem isto: No module named 'ftr')
├── configs/
├── scripts/
└── Data/dataset_FINAL.json
```

A célula abaixo **localiza a pasta automaticamente** (procurando por
`src/ftr`) e mostra um checklist do que está presente/faltando.

In [1]:
import os, sys, glob

# Monta o Drive (somente no Colab).
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print('Não está no Colab — usando caminho local.')


def _looks_like_project(p):
    return os.path.isdir(os.path.join(p, 'src', 'ftr'))


def _autodetect_root():
    candidates = ['/content/drive/MyDrive/Fato_Tese_Ruido']
    # procura qualquer pasta com src/ftr dentro do MyDrive (2 níveis)
    for base in ('/content/drive/MyDrive', '/content/drive/My Drive'):
        candidates += glob.glob(os.path.join(base, '**', 'src', 'ftr'), recursive=False)
        candidates += [os.path.dirname(os.path.dirname(p))
                       for p in glob.glob(os.path.join(base, '*', 'src', 'ftr'))]
    for c in candidates:
        if c and _looks_like_project(c):
            return c
    return candidates[0] if candidates else os.getcwd()


PROJECT_ROOT = _autodetect_root() if IN_COLAB else os.getcwd()
os.environ['FTR_PROJECT_ROOT'] = PROJECT_ROOT
if os.path.join(PROJECT_ROOT, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
print('PROJECT_ROOT =', PROJECT_ROOT)

# Itens OBRIGATÓRIOS (o código + os dados).
required = {'src/ftr': 'pasta (módulo ftr)', 'configs': 'pasta',
            'scripts': 'pasta', 'Data/dataset_FINAL.json': 'dataset'}
# Itens OPCIONAIS (o Drive às vezes converte .txt/.xlsx p/ Google Docs/Sheets,
# removendo a extensão — sem problema, instalamos as deps explicitamente abaixo).
optional = {'requirements.txt': 'arquivo (opcional)'}
print('\nChecklist da pasta do projeto:')
missing = []
for rel, kind in required.items():
    ok = os.path.exists(os.path.join(PROJECT_ROOT, rel))
    print(f'  [{"OK" if ok else "FALTA"}] {rel}  ({kind})')
    if not ok:
        missing.append(rel)
for rel, kind in optional.items():
    ok = os.path.exists(os.path.join(PROJECT_ROOT, rel))
    print(f'  [{"OK" if ok else "--"}] {rel}  ({kind})')
if missing:
    raise SystemExit('\nFALTAM itens OBRIGATÓRIOS acima. Suba a PASTA INTEIRA do '
                     'projeto para o Drive (não só Data/).')
print('\nOK — itens obrigatórios presentes. (requirements.txt é opcional.)')

Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/Fato_Tese_Ruido

Checklist da pasta do projeto:
  [OK] src/ftr  (pasta (módulo ftr))
  [OK] configs  (pasta)
  [OK] scripts  (pasta)
  [OK] Data/dataset_FINAL.json  (dataset)
  [--] requirements.txt  (arquivo (opcional))

OK — itens obrigatórios presentes. (requirements.txt é opcional.)


## 2. Instalar dependências

Instala as bibliotecas explicitamente (não depende do `requirements.txt`, que o
Drive às vezes converte para Google Docs). torch já vem no Colab.

In [2]:
if IN_COLAB:
    !pip install -q 'transformers==4.48.3' 'datasets==3.3.0' 'scikit-learn==1.6.1' \
        evaluate sentencepiece sacremoses 'accelerate>=1.3.0' openpyxl seaborn pyarrow
    print('\nDependências instaladas.')
else:
    print('Local: instale com  pip install -r requirements.txt')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 78.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 65.2 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.

Dependências instaladas.


## 3. Verificar GPU, ambiente e caminhos

In [3]:
import torch
from ftr.env import describe_environment, get_paths
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
import json
print(json.dumps(describe_environment(), indent=2, ensure_ascii=False))

CUDA disponível: False
{
  "in_colab": true,
  "drive_mounted": true,
  "project_root": "/content/drive/MyDrive/Fato_Tese_Ruido",
  "root_exists": true,
  "raw_dataset": "/content/drive/MyDrive/Fato_Tese_Ruido/Data/dataset_FINAL.json",
  "raw_dataset_exists": true,
  "python": "3.12.13",
  "platform": "linux"
}


## 4. Verificar que o dataset está no lugar

In [4]:
paths = get_paths()
print('Dataset bruto:', paths.raw_dataset)
assert paths.raw_dataset.exists(), (
    f'Dataset não encontrado em {paths.raw_dataset}. '
    'Suba Data/dataset_FINAL.json para o Drive.')
print('OK — dataset encontrado (%.1f MB).' % (paths.raw_dataset.stat().st_size/1e6))

Dataset bruto: /content/drive/MyDrive/Fato_Tese_Ruido/Data/dataset_FINAL.json
OK — dataset encontrado (259.0 MB).


## 5. Pronto!

Ambiente configurado. Agora você pode:
- Rodar `00_EDA.ipynb` para a análise exploratória;
- Rodar `01_experiment_template.ipynb` para treinar um modelo;
- Rodar `02_run_all_colab.ipynb` para treinar todos em sequência.

Antes do treino, prepare os dados (limpeza + anonimização + split):

In [5]:
# -W all exibe todos os warnings (importante para ver alertas de leakage)
!cd "{PROJECT_ROOT}" && python -W all scripts/01_prepare_data.py --config configs/_base.yaml

2026-06-02 17:57:57.826522: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-02 17:57:57.940801: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' depr